In [1]:
PATH_WORK_DIR = ".."

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.chdir(PATH_WORK_DIR)
print(f"DIRECTORY: {os.getcwd()}")

DIRECTORY: c:\Users\jayar\Desktop\바탕 화면\REPO\PROJECT\M2-PJT_STATS


# package

In [ ]:
import pandas as pd
import pymc as pm
import arviz as az

# Data

In [6]:
PATH = './data/housing.csv'
df = pd.read_csv(PATH)

In [7]:
df["YEARMONTH"] = pd.to_datetime(
    df["YEARMONTH"].astype(str),
    format="%Y-%m",
)

In [8]:
y          = df['PRICE'].values

REGION_IDX = df['REGION_IDX'].values
TIME_IDX   = df['TIME_IDX'].values
TREATMENT  = df['TREATMENT'].values
POST1      = df['PERIOD_1'].values
POST2      = df['PERIOD_2'].values

N_REGION   = df['REGION_IDX'].nunique()
N_CITY     = df['CITY_IDX'].nunique()
N_TIME     = df['TIME_IDX'].nunique()

REGION_TO_CITY = (
    df[['REGION_IDX', 'CITY_IDX']]
    .drop_duplicates()
    .sort_values('REGION_IDX')['CITY_IDX']
    .values
)

# Model Definition

In [9]:
model = pm.Model()

# Prior Definition

In [10]:
# REGION EFFECT ==========

with model:
    # baseline
    mu_alpha = pm.Normal(
        name="mu_alpha", 
        mu=0, 
        sigma=2,
    )
    sigma_city = pm.HalfNormal(
        name="sigma_city", 
        sigma=2,
    )

    # city level
    alpha_city = pm.Normal(
        name="alpha_city",
        mu=mu_alpha,
        sigma=sigma_city,
        shape=N_CITY,
    )
    sigma_region = pm.HalfNormal(
        name="sigma_region", 
        sigma=2,
    )

    # region level
    alpha_region = pm.Normal(
        name="alpha_region",
        mu=alpha_city[REGION_TO_CITY],
        sigma=sigma_region,
        shape=N_REGION,
    )

In [11]:
# TIME EFFECT ==========

with model:
    sigma_tau = pm.HalfNormal(
        name="sigma_tau", 
        sigma=2,
    )

    tau_raw = pm.GaussianRandomWalk(
        name="tau_raw",
        sigma=sigma_tau,
        shape=N_TIME,
    )

    tau = pm.Deterministic(
        name="tau",
        var=tau_raw - pm.math.mean(tau_raw),
    )

In [12]:
# ATT ==========

with model:
    delta1 = pm.Normal(
        name="delta1", 
        mu=0, 
        sigma=2,
    )
    delta2 = pm.Normal(
        name="delta2", 
        mu=0, 
        sigma=2,
    )

In [13]:
# y ==========

with model:
    REGION_EFFECT = alpha_region[REGION_IDX]
    TIME_EFFECT = tau[TIME_IDX]
    ATT = (
        delta1 * TREATMENT * POST1
        + delta2 * TREATMENT * POST2
    )

    mu = (
        REGION_EFFECT 
        + TIME_EFFECT 
        + ATT
    )
    sigma = pm.HalfNormal(
        name="sigma", 
        sigma=2,
    )

# Likelihood

In [14]:
with model:
    pm.Normal(
        name="obs",
        mu=mu,
        sigma=sigma,
        observed=y,
    )

# Fit

In [15]:
SEED = 42
DRAWS = 10000
TUNE = 5000
TARGET_ACCEPT = 0.9
CHAINS = 4

In [ ]:
with model:
    kwargs = dict(
        draws=DRAWS, 
        tune=TUNE, 
        target_accept=TARGET_ACCEPT,
        chains=CHAINS, 
        idata_kwargs={"log_likelihood": True}, 
        random_seed=SEED,
    )
    trace = pm.sample(**kwargs)

# Predict

In [ ]:
with model:
    ppc = pm.sample_posterior_predictive(trace)

# Save

In [ ]:
trace.extend(ppc)

In [ ]:
PATH = "./checkpoints/trace.nc"

kwargs = dict(
    data=trace, 
    filename=PATH,
)

az.to_netcdf(**kwargs)